In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_hnx = pd.read_excel(r"D:\Downloads\DSTC vòng 3\cleaned data\HNXINDEX_cleaned.xlsx")
df_hnx = df_hnx.drop("Unnamed: 0", axis =1)
df_hnx.info()

In [ ]:
df_hose = pd.read_excel(r"D:\Downloads\DSTC vòng 3\cleaned data\VNINDEX_cleaned.xlsx")
df_hose = df_hose.drop("Unnamed: 0", axis =1)
df_hose.info()

In [ ]:
df_hose["ticker"].nunique()

In [ ]:
df_upcom = pd.read_excel(r"D:\Downloads\DSTC vòng 3\cleaned data\UPCOM_cleaned.xlsx")
df_upcom = df_upcom.drop("Unnamed: 0", axis =1)
df_upcom.info()

In [ ]:
df_hose["exchange"] = "HOSE"
df_hnx["exchange"] = "HNX"
df_upcom["exchange"] = "UPCOM"

In [ ]:
df = pd.concat([df_hose, df_hnx, df_upcom], ignore_index=True)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(["exchange", "timestamp"])

In [ ]:
df["ticker"].nunique()

In [ ]:
df.reset_index(inplace=True)
df.head()

In [ ]:
df.drop("index", axis = 1, inplace = True)
df.head()

In [ ]:
df["value_trade"] = df["close"] * df["volume"]
df.info()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="exchange", y="volume", data=df, showfliers=False)
plt.title("So sánh khối lượng giao dịch (Volume)")
plt.yscale("log") 
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="exchange", y="volume", data=df, showfliers=True)
plt.title("So sánh khối lượng giao dịch (Volume)")
plt.yscale("log") 
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='exchange', y='return', data=df, showfliers=False) 
plt.title('Phân Bổ Tỷ Suất Sinh Lợi Hàng Ngày Giữa Các Sàn')
plt.axhline(0, color='red', linestyle='--') 
plt.ylabel('Tỷ suất sinh lợi (%)')
plt.xlabel('Sàn Giao Dịch')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='exchange', y='return', data=df, showfliers=True) 
plt.title('Phân Bổ Tỷ Suất Sinh Lợi Hàng Ngày Giữa Các Sàn')
plt.axhline(0, color='red', linestyle='--') 
plt.ylabel('Tỷ suất sinh lợi (%)')
plt.xlabel('Sàn Giao Dịch')
plt.show()

In [ ]:
def count_outliers(group):
    
    daily_return_series = group['return']

    Q1 = daily_return_series.quantile(0.25)
    Q3 = daily_return_series.quantile(0.75)
    IQR = Q3 - Q1

    # Xác định ngưỡng trên và ngưỡng dưới
    upper_bound = Q3 + 1.5 * IQR
    lower_bound = Q1 - 1.5 * IQR

    # Đếm số lượng outliers
    positive_outliers = (daily_return_series > upper_bound).sum()
    negative_outliers = (daily_return_series < lower_bound).sum()


    return pd.Series({
        'positive_outliers': positive_outliers,
        'negative_outliers': negative_outliers
    })


outlier_counts = df.groupby('exchange').apply(count_outliers)


print("Số Lượng Outliers Tăng/Giảm Trên Mỗi Sàn:")
print(outlier_counts)

In [ ]:
total_counts = df.groupby('exchange').size().rename('total_sessions')

final_stats = pd.concat([outlier_counts, total_counts], axis=1)


final_stats['positive_outlier_pct'] = (final_stats['positive_outliers'] / final_stats['total_sessions']) * 100
final_stats['negative_outlier_pct'] = (final_stats['negative_outliers'] / final_stats['total_sessions']) * 100
final_stats['total_outlier_pct'] = ((final_stats['positive_outliers'] + final_stats['negative_outliers']) / final_stats['total_sessions']) * 100

print("\nThống Kê Chi Tiết Về Tỷ Lệ Outliers:")
print(final_stats)

In [ ]:
df['intraday_range'] = (df['high'] - df['low']) / df['low']

plt.figure(figsize=(10, 6))
sns.boxplot(x='exchange', y='intraday_range', data=df, showfliers=False)
plt.title('So Sánh Biên Độ Dao Động Trong Phiên (Intraday)')
plt.ylabel('Biên độ dao động (%)')
plt.xlabel('Sàn Giao Dịch')
plt.show()

In [ ]:
df['intraday_range'] = (df['high'] - df['low']) / df['low']

plt.figure(figsize=(10, 6))
sns.boxplot(x='exchange', y='intraday_range', data=df, showfliers=True)
plt.title('So Sánh Biên Độ Dao Động Trong Phiên (Intraday)')
plt.ylabel('Biên độ dao động (%)')
plt.xlabel('Sàn Giao Dịch')
plt.show()

In [ ]:
def count_intraday_outliers(group):
    intraday_series = group['intraday_range']

    # Tính Q1, Q3, và IQR
    Q1 = intraday_series.quantile(0.25)
    Q3 = intraday_series.quantile(0.75)
    IQR = Q3 - Q1

    upper_bound = Q3 + 1.5 * IQR

    # Đếm số lượng phiên có biên độ dao động lớn hơn ngưỡng outlier
    outlier_count = (intraday_series > upper_bound).sum()

    return outlier_count

intraday_outlier_counts = df.groupby('exchange').apply(count_intraday_outliers).rename('intraday_outliers')


total_counts = df.groupby('exchange').size().rename('total_sessions')


intraday_stats = pd.concat([intraday_outlier_counts, total_counts], axis=1)

intraday_stats['outlier_pct'] = (intraday_stats['intraday_outliers'] / intraday_stats['total_sessions']) * 100

print("Thống Kê Số Lượng và Tỷ Lệ Outlier Của Biên Độ Dao Động Trong Phiên:")
print(intraday_stats)

In [ ]:
df['volatility_per_stock'] = df.groupby('ticker')['return'].rolling(window=30).std().reset_index(0, drop=True)

average_exchange_volatility = df.groupby(['timestamp', 'exchange'])['volatility_per_stock'].mean().unstack()

plt.figure(figsize=(15, 7))
average_exchange_volatility.plot(ax=plt.gca()) 
plt.title('Biến Động Trung Bình Của Cổ Phiếu Trên Các Sàn (Cửa sổ 30 ngày)')
plt.ylabel('Độ lệch chuẩn trung bình của TSSL')
plt.xlabel('Thời Gian')
plt.grid(True)
plt.legend(title='Sàn Giao Dịch')
plt.show()

In [ ]:
# def weighted_average_return(group):
#     weights = group['volume']
#     returns = group['return']
#     if weights.sum() == 0:
#         return 0
#     return (returns * weights).sum() / weights.sum()


# exchange_daily_return = df.groupby(['timestamp', 'exchange']).apply(weighted_average_return).unstack()

# exchange_daily_return.dropna(inplace=True)

exchange_daily_return = df.groupby(['timestamp', 'exchange'])['return'].mean().unstack()

correlation_matrix = exchange_daily_return.corr()

print("Ma Trận Tương Quan Giữa Các Sàn:")
print(correlation_matrix)



plt.figure(figsize=(8, 6))


sns.heatmap(
    correlation_matrix,
    annot=True,          
    cmap='coolwarm',     
    fmt=".2f",           
    vmin=-1, vmax=1      
)

plt.title('Ma Trận Tương Quan Tỷ Suất Sinh Lợi Giữa Các Sàn')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x="exchange", y="value_trade", data=df, showfliers=False)
plt.title("So sánh Phân Bổ Giá Trị Giao Dịch (Dòng Tiền) Mỗi Phiên")
plt.ylabel("Giá trị giao dịch (VND, thang đo log)")
plt.xlabel("Sàn Giao Dịch")
plt.yscale("log")  
plt.show()

In [ ]:
df['risk_score_annualized'] = df['volatility_per_stock'] * (252**0.5)

In [ ]:
if 'T1_dynamic' not in df.columns:
    df['T1_dynamic'] = df.groupby('timestamp')['risk_score_annualized'].transform(lambda x: x.quantile(0.30))
    df['T2_dynamic'] = df.groupby('timestamp')['risk_score_annualized'].transform(lambda x: x.quantile(0.70))
    # Thêm một ngưỡng nữa để tham khảo, ví dụ Q85 cho nhóm rủi ro cao hơn
    df['T_q85_dynamic'] = df.groupby('timestamp')['risk_score_annualized'].transform(lambda x: x.quantile(0.85))


# --- Lấy ra chuỗi thời gian của các ngưỡng ---
# Chúng ta chỉ cần lấy giá trị đầu tiên của mỗi ngày vì chúng giống nhau cho tất cả cổ phiếu
dynamic_thresholds = df.groupby('timestamp')[['T1_dynamic', 'T2_dynamic', 'T_q85_dynamic']].first()


# --- VẼ BIỂU ĐỒ ---
plt.figure(figsize=(15, 7))
dynamic_thresholds.plot(ax=plt.gca())
plt.title('Sự Thay Đổi Của Các Ngưỡng Rủi Ro (Phân Vị) Theo Thời Gian')
plt.ylabel('Giá Trị Ngưỡng (Annualized Volatility)')
plt.xlabel('Thời Gian')
plt.grid(True)
plt.legend(title='Ngưỡng Phân Vị')
plt.show()

In [ ]:
df['T1_dynamic'] = df.groupby('timestamp')['risk_score_annualized'].transform(lambda x: x.quantile(0.30))
df['T2_dynamic'] = df.groupby('timestamp')['risk_score_annualized'].transform(lambda x: x.quantile(0.70))

In [ ]:
import numpy as np

conditions = [
    df['risk_score_annualized'] < df['T1_dynamic'],
    (df['risk_score_annualized'] >= df['T1_dynamic']) & (df['risk_score_annualized'] < df['T2_dynamic']),
    df['risk_score_annualized'] >= df['T2_dynamic']
]

# Tạo một danh sách các nhãn tương ứng
labels = ['Rủi Ro Thấp', 'Rủi Ro Trung Bình', 'Rủi Ro Cao']

# Sử dụng np.select để gán nhãn một cách hiệu quả
df['risk_label'] = np.select(conditions, labels, default='Không Xác Định')

# Dọn dẹp dữ liệu: loại bỏ các hàng không có nhãn (những ngày đầu tiên chưa đủ dữ liệu)
df_final_for_ml = df[df['risk_label'] != 'Không Xác Định'].copy()


In [ ]:
df.tail(20)